# Day 1 · SONIC 해부하기 — 모델과 데이터의 '계약' 읽기
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day1_anatomy.ipynb)

**오늘의 결과물**: 공식 걷기 샘플 → *내가 만든* Encoder 입력(1762) → 64차원 토큰 시계열

| 시간 | 내용 |
|---|---|
| 0:00–0:15 | 환경 준비 · 목적지 미리보기 · 도구 연결 |
| 0:15–0:30 | Mission 1-1 ONNX 입출력 해부 |
| 0:30–0:50 | Mission 1-2 Encoder 1762차원 블록 표 |
| 0:50–1:10 | Mission 1-3 쿼터니언 순서 탐정 |
| 1:10–1:25 | Mission 1-4 30 Hz ↔ 50 Hz 시간 정렬 |
| 1:25–1:50 | Mission 1-5 Mode-2 Encoder 입력 조립 → 토큰 |
| 1:50–2:00 | 정리 · 프롬프트 로그 |

**규칙**: `sonic_lab/reference.py`, `sonic_lab/check.py`는 정답 파일이라 열지 않습니다. 근거는 `docs/01_contract.md`뿐입니다.
⚠️ 이 실습의 어떤 출력도 실물 로봇에 보내지 않습니다.

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"   # 에이전트형 도구 사용자는 내 Fork 주소로 바꾸세요
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

## 목적지 미리보기 (5분)
참조 구현으로 3일 뒤 도달할 결과를 먼저 봅니다: **사람 걷기 데이터 → SONIC → MuJoCo G1이 넘어지지 않고 걷기**.

In [ ]:
summary, arrays = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=250)
print({k: summary[k] for k in ["simulated_frames", "requested_frames", "completed", "stop_reason", "runtime_s"]})
render.show(render.skeleton_video(lab, arrays, "outputs/day1_preview.mp4", title="reference pipeline", seconds=5))

### 📝 프롬프트 로그 (이 셀을 더블클릭해 편집)
| # | 내가 한 요청 | AI 답 요약 | 내가 확인한 근거 | 채택? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

---
## Mission 1-1 · ONNX 입출력 해부 (15분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

- **목표**: `model_encoder.onnx`, `model_decoder.onnx`의 입력·출력 이름과 shape를 확인한다.
- **주어진 것**: `lab.assets.model_dir` (ONNX 2개 + `observation_config.yaml`)
- **완료 기준**: ✅ 셀 통과
- **생각해 볼 것**: `observation_config.yaml` 맨 위 주석의 *Total dimension* 값과 실제 Decoder 입력 길이를 비교해 보세요. 어느 쪽을 믿어야 할까요?

In [ ]:
# ✍️ Mission 1-1
def inspect_onnx(model_dir):
    """반환: (encoder 입력 길이, encoder 출력 길이, decoder 입력 길이, decoder 출력 길이)"""
    raise NotImplementedError

# from my_work.day1 import *    # 에이전트형 도구로 my_work/day1.py에 작성했다면 주석 해제

In [ ]:
# ✅ 확인
check.onnx_io(*inspect_onnx(lab.assets.model_dir))

---
## Mission 1-2 · Encoder 1762차원 블록 표 (20분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

- **목표**: `observation_config.yaml`을 읽어 Encoder 관측 14개 블록의 `(start, stop)` 표를 코드로 만든다.
- **함정**: YAML에는 **길이가 적혀 있지 않습니다.** 이름(`10frame`, `step5`, `lowerbody`, `vr_3point` …)과 계약서 3절로 길이를 정하고, 합이 ONNX 입력 길이와 같은지 `assert`로 확인하게 하세요.
- **완료 기준**: ✅ 셀 통과

In [ ]:
# ✍️ Mission 1-2
def build_encoder_slices(yaml_path):
    """반환: {관측 이름: (start, stop)} — YAML의 enabled 순서"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
slices = build_encoder_slices(lab.assets.model_dir / "observation_config.yaml")
check.encoder_slices(lab, slices)

---
## Mission 1-3 · 쿼터니언 순서 탐정 (20분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

공식 G1 샘플에는 루트 회전이 **두 가지 표현**으로 들어 있습니다: `pose_aa[:, 0]`(축-각 벡터)과 `root_rot`(쿼터니언).
실제 연구 과정(06번 노트북)에서 `root_rot`를 잘못된 성분 순서로 읽어 **최대 약 179°** 회전 오류가 난 적이 있습니다.

- **목표**: 두 표현을 비교해 `root_rot`가 `xyzw`인지 `wxyz`인지 **데이터로** 판정하는 함수를 만든다.
- **요청 팁**: "어느 쪽일 것 같아?"가 아니라 "두 가정 각각의 최대 각도 오차를 계산해서 비교하는 코드"를 요청하세요.
- **완료 기준**: ✅ 셀 통과 + 틀린 순서일 때 오차가 몇 도인지 출력

In [ ]:
# 데이터 불러오기 (제공)
from sonic_lab.reference import load_published_pickle
from sonic_lab.assets import SAMPLE
robot = load_published_pickle(lab.assets.sample_root / "robot_filtered/210531" / SAMPLE)
human = load_published_pickle(lab.assets.sample_root / "smpl_filtered" / SAMPLE)
print("G1  :", {k: np.shape(v) for k, v in robot.items()})
print("SMPL:", {k: np.shape(v) for k, v in human.items()})

In [ ]:
# ✍️ Mission 1-3
def detect_root_quat_order(robot_record):
    """반환: 'xyzw' 또는 'wxyz' (판정 근거가 되는 오차도 출력할 것)"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
check.root_quat_order(detect_root_quat_order(robot))

---
## Mission 1-4 · 30 Hz ↔ 50 Hz 시간 정렬 (15분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

- G1 참조는 30 fps, 사람(SMPL)은 50 fps이고 길이도 조금 다릅니다. 정책은 50 Hz로 돕니다.
- **목표**: 두 데이터 중 **짧은 쪽 길이**에 맞춘 50 Hz 시간축과, 그 시간에서 선형 보간한 G1 관절각 `(T, 29)`를 만든다.
- **완료 기준**: ✅ 셀 통과

In [ ]:
# ✍️ Mission 1-4
def resample_g1(robot_record, human_record, hz=50):
    """반환: (times (T,), g1_q50 (T, 29))"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
times, g1_q50 = resample_g1(robot, human)
check.resampled_motion(lab, times, g1_q50)

---
## Mission 1-5 · Mode-2 Encoder 입력 조립 → 토큰 (25분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

`lab.motion`은 이미 50 Hz로 정렬된 공식 샘플입니다(Mission 1-3/1-4를 못 끝냈어도 진행 가능).

| 필드 | shape | 의미 |
|---|---|---|
| `lab.motion.local24` | (T, 24, 3) | 루트 기준 SMPL 관절 좌표 |
| `lab.motion.human_root_xyzw` | (T, 4) | 인체 루트 회전 (scipy xyzw) |
| `lab.motion.wrists_il` | (T, 6) | G1 손목 6관절 (IsaacLab 23–28) |
| `lab.motion.times` | (T,) | 50 Hz 시간축 |

- **목표**: 계약서 3절대로 `frame` 시점의 1762차원 Mode-2 Encoder 입력을 만든다. `robot_xyzw`는 현재 로봇 기저 회전(xyzw), `alignment`는 scipy `Rotation`(헤딩 정렬 A)입니다.
- **완료 기준**: ✅ 셀 통과 → 토큰 시계열 그림

In [ ]:
# ✍️ Mission 1-5
from scipy.spatial.transform import Rotation as R

def build_encoder_obs_mode2(motion, frame, robot_xyzw, alignment):
    """반환: (1762,) float32"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
check.encoder_obs(lab, build_encoder_obs_mode2)

In [ ]:
# 내 Encoder 입력으로 만든 토큰 시계열 (처음 10초, 로봇이 참조와 같은 방향이라고 가정)
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
A = R.identity()
tokens = np.array([lab.policy.encode_obs(build_encoder_obs_mode2(lab.motion, t, lab.motion.human_root_xyzw[t], A))
                   for t in range(0, 500, 2)])
plt.figure(figsize=(11, 3))
plt.plot(lab.motion.times[0:500:2], tokens[:, :8]); plt.xlabel("time (s)"); plt.title("SONIC token dims 0–7 (walk sample)")
plt.show()

---
## 정리 (10분)
- 오늘 AI가 **틀렸던** 순간이 있었나요? 무엇으로 알았나요? → 프롬프트 로그에 기록
- 토큰 그래프가 걸음 주기를 따라 주기적으로 움직이나요? 그렇다고 해서 *입력이 맞다*고 할 수 있을까요?
- **Day 2 예고**: 오류 없이 돌아가지만 로봇이 넘어지는 'AI가 작성한' 파이프라인에서 버그 4개를 찾습니다.